# PowerMind RAG v2 — End-to-End Test Notebook

Tests in order:
1. **Config** — env vars load correctly
2. **Embedder** — local sentence-transformers works
3. **Pinecone** — connect + get index stats
4. **Ingestor** — chunk + embed + upsert a sample document
5. **LangGraph single-turn** — basic query
6. **LangGraph multi-turn** — conversation with history
7. **API endpoints** — hit the running FastAPI server via httpx
8. **Cleanup** — delete test vectors

> **Prerequisites**: Set `PINECONE_API_KEY` in `service/.env` before running.

## 0 · Setup — sys.path & env

In [1]:
import sys, os
from pathlib import Path

# Make both packages importable (run from project root)
ROOT = Path('.').resolve()
SERVICE_SRC = ROOT / 'service' / 'src'
for p in [str(SERVICE_SRC)]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Load .env from service/
from dotenv import load_dotenv
load_dotenv(ROOT / 'service' / '.env', override=True)

print('ROOT:', ROOT)
print('SERVICE_SRC:', SERVICE_SRC)
print('PINECONE_API_KEY set?', bool(os.getenv('PINECONE_API_KEY')))
print('GROQ_API_KEY    set?', bool(os.getenv('GROQ_API_KEY')))

ROOT: /Users/vatsalbateriwala/Developer/powermind/backend
SERVICE_SRC: /Users/vatsalbateriwala/Developer/powermind/backend/service/src
PINECONE_API_KEY set? False
GROQ_API_KEY    set? False


## 1 · Config

In [ ]:
from powermind_rag_v2.config import RagV2Config

cfg = RagV2Config.from_env()
print('Pinecone index  :', cfg.pinecone_index_name)
print('Pinecone ns     :', cfg.pinecone_namespace)
print('Embedding model :', cfg.embedding_model)
print('Embedding dim   :', cfg.embedding_dimension)
print('Groq model      :', cfg.groq_chat_model)
print('top_k           :', cfg.top_k)
print('score_threshold :', cfg.score_threshold)
print('chunk_size      :', cfg.chunk_size)
assert cfg.pinecone_api_key, 'PINECONE_API_KEY is missing — set it in service/.env'
assert cfg.groq_api_key,    'GROQ_API_KEY is missing — set it in service/.env'
print('\n✅ Config OK')

## 2 · Embedder (local)

In [ ]:
from powermind_rag_v2.embedder import build_embedder

embedder = build_embedder(cfg)

test_texts = [
    'PowerMind is a multimodal RAG system.',
    'Pinecone stores vector embeddings for fast similarity search.',
    'LangGraph orchestrates multi-step AI workflows.',
]
vecs = embedder.embed(test_texts)
print('Shape          :', vecs.shape)          # (3, 384)
print('Dtype          :', vecs.dtype)           # float32
print('L2 norms (≈1)  :', (vecs**2).sum(axis=1)**0.5)  # should all be ~1.0
assert vecs.shape == (3, 384), f'Expected (3, 384) got {vecs.shape}'
print('\n✅ Embedder OK')

## 3 · Pinecone — connect & stats

In [ ]:
from powermind_rag_v2.vector_store import PineconeStore

store = PineconeStore(cfg)

# This will create the index if it doesn't exist yet (takes ~30 s first time)
print('Connecting to Pinecone (index will be created if missing)...')
stats = store.describe()
print('Index stats:', stats)
print('\n✅ Pinecone connection OK')

## 4 · Ingestor — create sample doc & ingest

In [ ]:
import tempfile, textwrap
from pathlib import Path
from powermind_rag_v2.ingestor import DocumentIngestor

SAMPLE_TEXT = textwrap.dedent("""
    PowerMind is a next-generation multimodal RAG (Retrieval-Augmented Generation) platform.
    It uses Pinecone for scalable vector storage and LangGraph for multi-step orchestration.
    The system supports PDF, TXT, and DOCX document ingestion.
    
    Key features:
    - Pinecone serverless index with cosine similarity search.
    - LangGraph StateGraph with 5 nodes: rewrite_query, retrieve, grade_relevance, generate, fallback.
    - Groq LLM (llama-3.3-70b-versatile) for generation, rewriting, and relevance grading.
    - Local sentence-transformers (all-MiniLM-L6-v2) for embedding — no cloud API needed.
    
    The retrieval pipeline first rewrites the user query into a standalone question,
    then fetches top-k chunks from Pinecone, grades each chunk for relevance using the LLM,
    and finally generates a grounded, cited answer.
    If no relevant chunks are found, a friendly fallback message is returned.
""").strip()

# Write to a temp .txt file
tmp = Path(tempfile.mktemp(suffix='.txt'))
tmp.write_text(SAMPLE_TEXT)
print('Temp file:', tmp)

ingestor = DocumentIngestor(cfg)
doc_id = ingestor.ingest(tmp, metadata={'source': 'notebook_test'})

print('Document ID:', doc_id)
tmp.unlink()  # clean up temp file

import time; time.sleep(3)  # wait for Pinecone to index
stats_after = store.describe()
print('Stats after ingest:', stats_after)
print('\n✅ Ingestion OK')

## 5 · LangGraph — single-turn query

In [ ]:
from powermind_rag_v2.graph import build_graph, RagState

graph = build_graph(cfg)

state: RagState = {
    'query': 'What LLM does PowerMind use for generation?',
    'history': [],
    'standalone_q': '',
    'chunks': [],
    'graded_chunks': [],
    'answer': '',
    'sources': [],
    'is_fallback': False,
}

result = graph.invoke(state)

print('Standalone Q  :', result['standalone_q'])
print('Chunks found  :', len(result['chunks']))
print('Graded chunks :', len(result['graded_chunks']))
print('Is fallback   :', result['is_fallback'])
print()
print('=== ANSWER ===')
print(result['answer'])
print()
print('=== SOURCES ===')
for s in result['sources']:
    print(f"  {s['citation']} — {s['filename']} (page {s['page_number']}, score {s['score']:.3f})")
print('\n✅ Single-turn query OK')

## 6 · LangGraph — multi-turn conversation

In [ ]:
# Simulate a 3-turn conversation
history = []

questions = [
    'What is PowerMind?',
    'What are its key features?',
    'How does the retrieval pipeline work exactly?',
]

for i, q in enumerate(questions, 1):
    print(f'\n--- Turn {i} ---')
    print(f'User: {q}')
    
    state = {
        'query': q,
        'history': history.copy(),
        'standalone_q': '',
        'chunks': [],
        'graded_chunks': [],
        'answer': '',
        'sources': [],
        'is_fallback': False,
    }
    result = graph.invoke(state)
    
    answer = result['answer']
    print(f'Assistant ({"FALLBACK" if result["is_fallback"] else "RAG"}): {answer[:300]}...' if len(answer) > 300 else f'Assistant: {answer}')
    
    # Append to history for next turn
    history.append({'role': 'user', 'content': q})
    history.append({'role': 'assistant', 'content': answer})

print('\n✅ Multi-turn conversation OK')

## 7 · Fallback — query with no matching context

In [ ]:
state = {
    'query': 'What is the capital of France?',  # unrelated to ingested doc
    'history': [],
    'standalone_q': '',
    'chunks': [],
    'graded_chunks': [],
    'answer': '',
    'sources': [],
    'is_fallback': False,
}

result = graph.invoke(state)
print('Is fallback:', result['is_fallback'])
print('Answer     :', result['answer'])
print('\n✅ Fallback behavior OK')

## 8 · API Endpoints — live FastAPI server

> **Start the backend first:**
> ```bash
> cd backend && uv run fastapi dev app/main.py
> ```

In [ ]:
import httpx, json

BASE = 'http://127.0.0.1:8000'

# Health check
r = httpx.get(f'{BASE}/health', timeout=5)
print('Health:', r.json())

# Pinecone index stats via API
r = httpx.get(f'{BASE}/ingest/v2/store/stats', timeout=15)
print('Store stats:', r.json())

print('\n✅ API server reachable')

In [ ]:
# Upload a file via /ingest/v2/file
import tempfile, textwrap

DOC = textwrap.dedent("""
    LangGraph is a framework for building stateful, multi-actor applications with LLMs.
    It extends LangChain with graph-based orchestration primitives.
    Each node in the graph receives the shared state, transforms it, and passes it on.
    Conditional edges allow dynamic branching based on runtime state values.
""").strip()

tmp2 = Path(tempfile.mktemp(suffix='.txt'))
tmp2.write_text(DOC)

with open(tmp2, 'rb') as f:
    r = httpx.post(
        f'{BASE}/ingest/v2/file',
        files={'file': ('langgraph_doc.txt', f, 'text/plain')},
        timeout=30,
    )
tmp2.unlink()

print('Status:', r.status_code)
file_record = r.json()
print('File record:', json.dumps(file_record, indent=2))
file_id = file_record['id']
print('\n✅ File uploaded (ingestion running in background)')

In [ ]:
import time

# Poll until completed
for attempt in range(12):
    r = httpx.get(f'{BASE}/ingest/v2/files/{file_id}', timeout=10)
    rec = r.json()
    status = rec['status']
    print(f'  [{attempt+1}] status = {status}')
    if status in ('completed', 'failed'):
        break
    time.sleep(5)

print('Final record:', json.dumps(rec, indent=2))
assert status == 'completed', f'Ingestion failed: {rec.get("error")}'
print('\n✅ Background ingestion completed')

In [ ]:
# Create a v2 chat session and send messages
r = httpx.post(f'{BASE}/chat/v2/sessions', json={'title': 'Notebook Test'}, timeout=10)
session = r.json()
session_id = session['id']
print('Session:', session)

# First message
r = httpx.post(
    f'{BASE}/chat/v2/sessions/{session_id}/messages',
    json={'content': 'What is LangGraph and how does it use conditional edges?'},
    timeout=60,
)
turn1 = r.json()
print('\nUser   :', turn1['user_message']['content'])
print('Reply  :', turn1['assistant_message']['content'])
print('Sources:', turn1['assistant_message']['meta'].get('sources', []))

In [ ]:
# Second message — uses conversation history
r = httpx.post(
    f'{BASE}/chat/v2/sessions/{session_id}/messages',
    json={'content': 'How does that compare to standard LangChain?'},
    timeout=60,
)
turn2 = r.json()
print('User  :', turn2['user_message']['content'])
print('Reply :', turn2['assistant_message']['content'])
print('\n✅ Multi-turn API chat OK')

In [ ]:
# List all messages in the session
r = httpx.get(f'{BASE}/chat/v2/sessions/{session_id}/messages', timeout=10)
messages = r.json()
print(f'Total messages in session: {len(messages)}')
for m in messages:
    print(f"  [{m['role']:9s}] {m['content'][:80]}")

## 9 · Cleanup — delete test vectors & session

In [ ]:
# Delete the file record + Pinecone vectors
r = httpx.delete(f'{BASE}/ingest/v2/files/{file_id}', timeout=15)
print('Delete file status:', r.status_code)  # 204

# Delete the chat session
r = httpx.delete(f'{BASE}/chat/v2/sessions/{session_id}', timeout=10)
print('Delete session status:', r.status_code)  # 204

# Also clean up notebook-test document (from section 4)
ingestor.delete(doc_id)
print('Deleted notebook-test document from Pinecone:', doc_id)

stats_final = store.describe()
print('Final Pinecone stats:', stats_final)
print('\n✅ Cleanup complete')

---
## Summary

| # | Test | What it checks |
|---|---|---|
| 1 | Config | All env vars loaded, API keys present |
| 2 | Embedder | Local model produces (N, 384) float32, L2-normalised |
| 3 | Pinecone | Index created/connected, stats returned |
| 4 | Ingestor | Text chunked → embedded → upserted to Pinecone |
| 5 | Graph single-turn | Query rewrite → retrieval → grading → generation |
| 6 | Graph multi-turn | History passed across 3 turns |
| 7 | Fallback | Unrelated query triggers fallback node |
| 8 | API ingest | POST /ingest/v2/file + status polling |
| 8 | API chat | POST /chat/v2/sessions + 2-turn conversation |
| 9 | Cleanup | Vectors + DB records deleted |